# Unknown Function Approximation

##### Author: [Radoslav Neychev](https://www.linkedin.com/in/radoslav-neychev/), telegram: [@rads_ai](https://t.me/rads_ai)

In this assignment, you will face an unknown dependency. Your main task is to **build two best models** that minimize mean squared error (MSE):
1. The first model has no restrictions.
2. The second model must be **linear**, i.e., it must be a linear combination of features plus a bias term: $\boldsymbol{w}^{\top}\boldsymbol{x} + b$. At the same time, __you may use basic mathematical operations to transform features__: np.exp, np.log, np.pow (the full list is available in the [documentation](https://numpy.org/doc/stable/reference/routines.math.html)), as well as linear operations on them (sum, multiplication by a number, etc.). To transform features, you will need to write the function `my_transformation`. __The number of parameters (weights) used by the second model must not exceed 15 (including the bias term).__

We strongly recommend writing the code "from scratch", only occasionally looking at ready-made examples, rather than simply copy-pasting them. This will help you write code more confidently in the future.

In [ ]:
import os
import json

import numpy as np
from matplotlib import pyplot as plt
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split

The data is loaded below. If loading does not work, download the file `hw_final_open_data.npy` yourself and place it in the same directory as the notebook.

In [ ]:
!wget https://raw.githubusercontent.com/girafe-ai/ml-course/23f_yandex_ml_trainings/homeworks/assignment_final/hw_final_open_data.npy -O hw_final_open_data.npy
!wget https://raw.githubusercontent.com/girafe-ai/ml-course/23f_yandex_ml_trainings/homeworks/assignment_final/hw_final_open_target.npy -O hw_final_open_target.npy

In [ ]:
assert os.path.exists('hw_final_open_data.npy'), 'Please, download `hw_final_open_data.npy` and place it in the working directory'
assert os.path.exists('hw_final_open_target.npy'), 'Please, download `hw_final_open_target.npy` and place it in the working directory'
data = np.load('hw_final_open_data.npy', allow_pickle=False)
target = np.load('hw_final_open_target.npy', allow_pickle=False)

The split into `train` and `val` is optional and is provided for your convenience.

In [ ]:
train_x, valid_x, train_y, valid_y = train_test_split(data, target, test_size=0.3)

### Model #1
As a reminder, in the first part of the assignment your main task is to obtain the best possible result without model restrictions. Only model predictions will be submitted.

An example using Random Forest is available below.

In [ ]:
rf = RandomForestRegressor()
rf.fit(train_x, train_y)

print(
    f'train mse =\t {mean_squared_error(np.round(rf.predict(train_x), 2), np.round(train_y)):.5f}',
    f'validation mse = {mean_squared_error(np.round(rf.predict(valid_x)), np.round(valid_y)):.5f}',
    sep='\n'
)

##### Submitting the First Part of the Contest
Download the file `hw_final_closed_data.npy` (the link is available on the assignment page). If you use an sklearn-compatible model, you can use the `get_predictions` function to generate the submission. Otherwise, rewrite the function for your model and run the code under the next cell to generate the submission.

In [ ]:
!wget https://raw.githubusercontent.com/girafe-ai/ml-course/23f_yandex_ml_trainings/homeworks/assignment_final/hw_final_closed_data.npy -O hw_final_closed_data.npy

In [ ]:
assert os.path.exists('hw_final_closed_data.npy'), 'Please, download `hw_final_closed_data.npy` and place it in the working directory'
closed_data = np.load('hw_final_closed_data.npy', allow_pickle=False)

If necessary, transform the data. Save the transformed object-feature matrix to the variable `closed_data`.

In [ ]:
# optional transformations

In [ ]:
def get_predictions(model, eval_data, step=10):
    predicted_values = model.predict(eval_data)
    return predicted_values

Please note that predictions are rounded to hundredths!

In [ ]:
predicted_values = np.round(get_predictions(model=rf, eval_data=closed_data), 2)

assert predicted_values.shape == (closed_data.shape[0], ) # predictions should be just one-dimensional array

In [ ]:
# do not change the code in the block below
# __________start of block__________
def float_list_to_comma_separated_str(_list):
    _list = list(np.round(np.array(_list), 2))
    return ','.join([str(x) for x in _list])

submission_dict = {
    'predictions': float_list_to_comma_separated_str(predicted_values)
}
with open('submission_dict_final_p01.json', 'w') as iofile:
    json.dump(submission_dict, iofile)
    
print('File saved to `submission_dict_final_p01.json`')
# __________end of block__________

### Model #2
The function `my_transformation` takes an object-feature matrix (`numpy.ndarray` of type `np.float`) as input and transforms it into a new matrix. This function may use only NumPy operations and arithmetic operations.

An example function is available below. It only adds a new feature equal to the product of the first and second original features (counting from zero).

In [ ]:
def my_transformation(feature_matrix: np.ndarray):
    new_feature_matrix = np.zeros((feature_matrix.shape[0], feature_matrix.shape[1]+1))
    new_feature_matrix[:, :feature_matrix.shape[1]] = feature_matrix
    new_feature_matrix[:, -1] = feature_matrix[:, 0] * feature_matrix[:, 1]
    return new_feature_matrix

In [ ]:
transformed_train_x = my_transformation(train_x)

In [ ]:
lr = Ridge()
lr.fit(transformed_train_x, train_y)

print(
    f'train mse =\t {mean_squared_error(lr.predict(transformed_train_x), train_y):.5f}',
    f'validation mse = {mean_squared_error(lr.predict(my_transformation(valid_x)), valid_y):.5f}',
    sep='\n'
)

Please note that the parameters of the linear model will be rounded to __four decimal places__. This should not significantly affect prediction quality:

In [ ]:
original_predictions = lr.predict(transformed_train_x)
rounded_predictions = transformed_train_x.dot(np.round(lr.coef_, 4)) + np.round(lr.intercept_, 4)


assert np.allclose(original_predictions, rounded_predictions, atol=1e-3)

Your model parameters:

In [ ]:
w_list = list(np.round(lr.coef_, 4))
print(f'w = {list(np.round(lr.coef_, 4))}\nb = {np.round(lr.intercept_, 4)}')

As a reminder, your model must not use more than 15 parameters (14 weights plus the bias term).

In [ ]:
assert len(w_list) + 1 <= 15

##### Submitting the Second Part of the Contest
To submit, it is enough to send the `my_transformation` function and your model parameters to the contest in Task #2. An example submission is available below. Importing `numpy` is also necessary.

In [ ]:
# __________example_submission_start__________
import numpy as np
def my_transformation(feature_matrix: np.ndarray):
    new_feature_matrix = np.zeros((feature_matrix.shape[0], feature_matrix.shape[1]+1))
    new_feature_matrix[:, :feature_matrix.shape[1]] = feature_matrix
    new_feature_matrix[:, -1] = feature_matrix[:, 0
    ] * feature_matrix[:, 1]
    return new_feature_matrix

w_submission = [-0.0027, -0.2637, 0.0, -0.1134, -0.0165, -0.9329, 0.0, 0.1293]
b_submission = 1.1312
# __________example_submission_end__________

This completes the assignment. Congratulations!